In [3]:
import pandas as pd
import sys
import os

if "Desktop" in os.getcwd():
    base_path = "/Users/sakdag/Desktop/masters/weakly-supervised-fake-news-classification"
else:
    base_path = "/home/sakdag/weakly-supervised-fake-news-classification"

print(base_path)
sys.path.append(base_path)
    
from src.weak_labeling.model_based.xgboost.xgboost_model import preprocess_for_xgboost_train
from src.utils.path_utils import get_project_root
from src.preprocessing.consts import Datasets
from src.preprocessing.read_raw_data import read_raw_data
from src.preprocessing.read_raw_data import read_csv
from src.weak_labeling.consts.globals import NLP

from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier

/home/sakdag/weakly-supervised-fake-news-classification


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, train_test_split
from src.preprocessing.tfidf_vectorizer import load_tfidf_vectorizer
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from src.preprocessing.cleanup import clean_text

In [5]:
injected_df: pd.DataFrame = read_csv(Datasets.NELA_ELECTIONS_SUBSET_2020.injected_path)

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-elections-subset-2020-sampled.csv
Read dataset, dataframe size: 2000


In [24]:
x = injected_df["title"]
y = injected_df["label"]
x_cleaned = x.apply(lambda t : clean_text(t))
X_train, X_test, y_train, y_test = train_test_split(
    x_cleaned, y, test_size=0.2, random_state=42
)

In [26]:
tfidf_vectorizer = load_tfidf_vectorizer("title")
train_vectors = tfidf_vectorizer.transform(X_train)
test_vectors = tfidf_vectorizer.transform(X_test)

Loading TFIDF vectorizer from path: /home/sakdag/weakly-supervised-fake-news-classification/model/weak_labeling/tfidf_vectorizer/generic-title.pkl


In [27]:
params = {"learning_rate": 0.15, "max_depth": 9, "n_estimators": 1000}

xgb = XGBClassifier(
    **params,
    eval_metric="auc",
    use_label_encoder=False,
    objective="binary:logistic",
)
xgb.fit(train_vectors, y_train)

xgb_pred = xgb.predict(test_vectors)
xgb_score = accuracy_score(y_test, xgb_pred)
xgb_roc = roc_auc_score(y_test, xgb_pred)
print("The accuracy of XGBOOST is: %0.2f" % xgb_score)
print("The roc_auc score of XGBOOST is: %0.2f" % xgb_roc)
cr = classification_report(y_test, xgb_pred)
print(cr)

The accuracy of XGBOOST is: 0.67
The roc_auc score of XGBOOST is: 0.67
              precision    recall  f1-score   support

           0       0.65      0.71      0.68       199
           1       0.68      0.63      0.65       201

    accuracy                           0.67       400
   macro avg       0.67      0.67      0.67       400
weighted avg       0.67      0.67      0.67       400



In [28]:
test_df: pd.DataFrame = read_csv(Datasets.NELA_COVID_2021.injected_path)

test_df_x = test_df["title"]
test_df_y = test_df["label"]
test_df_x_cleaned = test_df_x.apply(lambda t : clean_text(t))

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-covid-2021-sampled.csv
Read dataset, dataframe size: 50000


In [29]:
test_df_vectors = tfidf_vectorizer.transform(test_df_x_cleaned)

In [30]:
xgb_pred = xgb.predict(test_df_vectors)
xgb_score = accuracy_score(test_df_y, xgb_pred)
xgb_roc = roc_auc_score(test_df_y, xgb_pred)
print("The accuracy of XGBOOST is: %0.2f" % xgb_score)
print("The roc_auc score of XGBOOST is: %0.2f" % xgb_roc)
cr = classification_report(test_df_y, xgb_pred)
print(cr)

The accuracy of XGBOOST is: 0.59
The roc_auc score of XGBOOST is: 0.59
              precision    recall  f1-score   support

           0       0.61      0.48      0.54     25000
           1       0.57      0.69      0.63     25000

    accuracy                           0.59     50000
   macro avg       0.59      0.59      0.58     50000
weighted avg       0.59      0.59      0.58     50000



In [21]:
datasets = [
    Datasets.NELA_ELECTIONS_2020,
    Datasets.NELA_COVID_2021,
#     Datasets.NELA_COVID_2020,
#     Datasets.NELA_CAPITOL_2021,
    Datasets.NELA_ELECTIONS_SUBSET_2020,
]
big_df = read_csv(datasets[0].injected_path)
for dataset in datasets[1:]:
    big_df = pd.concat([big_df, read_csv(dataset.injected_path)])
print(len(big_df))

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-elections-2020-sampled.csv
Read dataset, dataframe size: 50000
Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-covid-2021-sampled.csv
Read dataset, dataframe size: 50000
Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-elections-subset-2020-sampled.csv
Read dataset, dataframe size: 2000
102000


In [23]:
from src.preprocessing.tfidf_vectorizer import save_tfidf_vectorizer


save_tfidf_vectorizer(big_df, "title")

Saving TFIDF vectorizer to path: /home/sakdag/weakly-supervised-fake-news-classification/model/weak_labeling/tfidf_vectorizer/generic-title.pkl


In [ ]:
from src.weak_labeling.model_based.xgboost.xgboost_model import train_xgb_model

In [ ]:
import datetime

train_xgb_model(Datasets.NELA_ELECTIONS_SUBSET_2020, save_identifier="test_jupyter", optimize=False)

In [ ]:
dataset = Datasets.NELA_ELECTIONS_SUBSET_2020
save_identifier = "test_jupyter"
optimize = False

In [ ]:
df: pd.DataFrame = read_csv(dataset.injected_path)
# roberta_train, roberta_test, y_test, y_train = preprocess_for_xgboost_train(df)

In [ ]:
# print(len(roberta_train))
# print(len(roberta_train[0]))

In [ ]:
x = df["content"]
y = df["label"]

In [2]:
from gensim import utils
import gensim.parsing.preprocessing as gsp

filters = [
           gsp.strip_tags, 
           gsp.strip_punctuation,
           gsp.strip_multiple_whitespaces,
           gsp.strip_numeric,
           gsp.remove_stopwords, 
           gsp.strip_short, 
           gsp.stem_text
          ]

def clean_text(s):
    s = s.lower()
    s = utils.to_unicode(s)
    for f in filters:
        s = f(s)
    return s

In [ ]:
test_df: pd.DataFrame = read_csv(Datasets.NELA_COVID_2021.injected_path)

In [ ]:
# tfidf_vectorizer = TfidfVectorizer()
# all_df_x = x.apply(lambda t : clean_text(t))
# tfidf_vectorizer.fit(all_df_x)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    all_df_x, y, test_size=0.2, random_state=42
)

In [ ]:
train_vectors = tfidf_vectorizer.transform(X_train)
test_vectors = tfidf_vectorizer.transform(X_test)

In [ ]:
test_vectors

In [ ]:
train_vectors

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

if optimize:
    params = get_best_xgb_parameters(roberta_train, y_train)
else:
    params = {"learning_rate": 0.15, "max_depth": 9, "n_estimators": 1000}

xgb = XGBClassifier(
    **params,
    eval_metric="auc",
    use_label_encoder=False,
    objective="binary:logistic",
)
xgb.fit(train_vectors, y_train)

xgb_pred = xgb.predict(test_vectors)
xgb_score = accuracy_score(y_test, xgb_pred)
xgb_roc = roc_auc_score(y_test, xgb_pred)
print("The accuracy of XGBOOST is: %0.2f" % xgb_score)
print("The roc_auc score of XGBOOST is: %0.2f" % xgb_roc)
cr = classification_report(y_test, xgb_pred)
print(cr)

In [ ]:
test_df_x = test_df["content"]
test_df_y = test_df["label"]
test_df_x_cleaned = test_df_x.apply(lambda t : clean_text(t))
test_df_vectors = tfidf_vectorizer.transform(test_df_x_cleaned)

In [ ]:
test_df_vectors

In [ ]:
xgb_pred = xgb.predict(test_df_vectors)
xgb_score = accuracy_score(test_df_y, xgb_pred)
xgb_roc = roc_auc_score(test_df_y, xgb_pred)
print("The accuracy of XGBOOST is: %0.2f" % xgb_score)
print("The roc_auc score of XGBOOST is: %0.2f" % xgb_roc)
cr = classification_report(test_df_y, xgb_pred)
print(cr)

In [ ]:
# TFIDF vectorizer write to common loc

In [4]:
datasets = [
    Datasets.NELA_ELECTIONS_2020,
    Datasets.NELA_COVID_2021,
#     Datasets.NELA_COVID_2020,
#     Datasets.NELA_CAPITOL_2021,
    Datasets.NELA_ELECTIONS_SUBSET_2020,
]
df = read_csv(datasets[0].injected_path)
for dataset in datasets[1:]:
    df = pd.concat([df, read_csv(dataset.injected_path)])
print(len(df))

Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-elections-2020-sampled.csv
Read dataset, dataframe size: 50000
Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-covid-2021-sampled.csv
Read dataset, dataframe size: 50000
Reading dataset from path: /home/sakdag/weakly-supervised-fake-news-classification/data/feature_injected/nela/nela-elections-subset-2020-sampled.csv
Read dataset, dataframe size: 2000
102000


In [5]:
tfidf_vectorizer_content = TfidfVectorizer()
content = df["content"]
content_cleaned = content.apply(lambda t : clean_text(t))

NameError: name 'tfidf_vectorizer' is not defined

In [28]:
word_counts = {}
for content in content_cleaned:
    for word in content.split():
        if word in word_counts.keys():
            word_counts[word] += 1
        else:
            word_counts[word] = 1

In [29]:
print(len(word_counts.keys()))

226129


In [30]:
count_1 = 0
count_less_than_5 = 0
count_less_than_10 = 0
for key in word_counts.keys():
    val = word_counts[key]
    if val == 1:
        count_1 += 1
    if val <= 5:
        count_less_than_5 += 1
    if val <= 10:
        count_less_than_10 += 1

In [31]:
print(count_1)
print(count_less_than_5)
print(count_less_than_10)

105454
166246
182730


In [33]:
tfidf_vectorizer_content = TfidfVectorizer(max_features=5000)
tfidf_vectorizer_content.fit(content_cleaned)

TfidfVectorizer(max_features=5000)

In [34]:
dummy = pd.DataFrame(["some text here"])
dummy_cleaned = dummy[0].apply(lambda t : clean_text(t))
dummy_vectors = tfidf_vectorizer_content.transform(dummy_cleaned)
print(dummy_vectors)

  (0, 4478)	1.0


In [35]:
import pickle
from src.weak_labeling.model_based.utils import get_any_file_name

tfidf_vect_file_name = get_any_file_name(
    "tfidf_vectorizer", f"generic-content-5000"
)
print(tfidf_vect_file_name)
pickle.dump(tfidf_vectorizer_content, open(tfidf_vect_file_name, "wb"))

/home/sakdag/weakly-supervised-fake-news-classification/model/weak_labeling/tfidf_vectorizer/generic-content-5000.pkl
